# Module 2 · Lesson 05: Prompt Evaluation

How do you know if your prompt is **good**? In production, we use **LLM-as-Judge**
to automatically evaluate prompt quality.

## What you will learn
1. Why prompt evaluation matters
2. **LLM-as-Judge** — using one model to evaluate another
3. Building a **scoring rubric**
4. **A/B testing** prompts
5. Batch evaluation for consistency

In [1]:
# ── Setup ──────────────────────────────────────────────
import os, json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0.7, max_tokens=400):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
print("✅ Ready")

✅ Ready


---
## 1. LLM-as-Judge Pattern

Use a **stronger model** (or the same model with a judge prompt) to score outputs.

```
Prompt A → Model → Response A ─┐
                                ├─→ Judge (LLM) → Score
Rubric ─────────────────────────┘
```

In [2]:
def evaluate_response(question: str, response: str, criteria: str) -> dict:
    """Use LLM to evaluate this AI response on a scale of 1-10"""
 
    judge_prompt = f"""Evaluate this AI response on a scale of 1-10.
 
Question: {question}
Response: {response}
 
Criteria: {criteria}
 
Return JSON: {{"score":<1-10>, "reasoning":"<brief explanation>"}}"""
 
    result = ask(judge_prompt, temperature=0)
 
    try:
        # Clean up markdown code blocks if present
        text = result.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except:
        return {"score": 0, "reasoning": result}
 
# Test it
question = "What is recursion in programming?"
response = ask(question)

In [3]:
eval_response = evaluate_response(
    question,
    response,
    "Accuracy, clarity, use of examples, appropiate lenght"
)
 
display(Markdown(f"### Response:\n{response}\n\n ### Evaluation:\n **Score:** {eval_response.get('score', 'N/A')}\n **Reasoning:**{eval_response.get('reasoning', 'N/A')}"))

### Response:
Recursion in programming refers to a technique where a function calls itself in order to solve a problem. This method is often used to break down complex problems into simpler subproblems. A recursive function typically has two main components:

1. **Base Case**: This is the condition under which the function stops calling itself. It serves as a termination point to prevent infinite recursion, which would lead to a stack overflow error.

2. **Recursive Case**: This is the part of the function where the function calls itself with modified arguments, moving closer to the base case.

### Example of Recursion

A classic example of recursion is the calculation of the factorial of a number \( n \), denoted as \( n! \). The factorial of a non-negative integer \( n \) is the product of all positive integers less than or equal to \( n \). The factorial can be defined recursively as follows:

- **Base Case**: \( 0! = 1 \)
- **Recursive Case**: \( n! = n \times (n - 1)! \) for \( n > 0 \)

Here is how you might implement this in Python:

```python
def factorial(n):
    if n == 0:  # Base case
        return 1
    else:       # Recursive case
        return n * factorial(n - 1)

# Example usage
print(factorial(5))  # Output: 120
```

### Advantages of Recursion
- **Simplicity**: Recursive solutions can be more straightforward and easier to understand compared to iterative solutions, especially for problems that have a natural recursive structure (like tree traversals).
- **Reduced Code Size**: Recursive functions can often be written with fewer lines of code.

### Disadvantages of Recursion
- **Performance**: Recursive functions can lead to high memory usage due to the call stack. Each recursive call consumes stack space, and deep recursion can lead to stack

 ### Evaluation:
 **Score:** 9
 **Reasoning:**The response accurately defines recursion, clearly explains its components (base case and recursive case), and provides a relevant example with a Python implementation. It also discusses the advantages and disadvantages of recursion, which adds depth to the explanation. The only minor improvement could be a more concise conclusion or summary to enhance clarity.

---
## 2. A/B Testing Prompts

Compare two different prompts on the **same question**:

In [5]:
question = "Explain what Docker is to a junior developer"
 
# Simple prompt
response_a = ask(question)
 
# Prompt: PCTF
response_b = ask(
    question,
    system="You are a senior DevOps engineer. Use real-world analogy. Keep it under 100 words."
)
 
criteria = "Clarity for begineer, use of analogies, conciseness, actionable information"
 
eval_a = evaluate_response(question, response_a, criteria)
eval_b = evaluate_response(question, response_b, criteria)
 
md = f"""### A/B Test Results
 
| | Prompt A (Simple) | Prompt B (PCTF) |
|---|---|---|
| **Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Words** | {len(response_a.split())} | {len(response_b.split())} |
 
#### Prompt A Response
{response_a[:300]}{'...' if len(response_a)>300 else ''}
 
#### Prompt B Response
{response_b[:300]}{'...' if len(response_b)>300 else ''}
"""
display(Markdown(md))
 
winner = "A" if eval_a.get('score',0) > eval_b.get('score',0) else "B"
print(f"\n Winner: Prompt {winner}")

### A/B Test Results

| | Prompt A (Simple) | Prompt B (PCTF) |
|---|---|---|
| **Score** | 9/10 | 9/10 |
| **Words** | 322 | 78 |

#### Prompt A Response
Sure! Docker is a platform that allows developers to automate the deployment of applications in lightweight, portable containers. Here’s a breakdown of what that means:

1. **Containers**: Think of a container as a package that includes everything needed to run an application—code, libraries, depend...

#### Prompt B Response
Think of Docker like a shipping container for software. Just as shipping containers standardize how goods are transported, Docker packages applications and all their dependencies into a single unit. This ensures that your app runs consistently across different environments, whether on your laptop, a...



 Winner: Prompt B


In [6]:
question = "Explain what Docker is to a junior developer"

# Simple prompt
response_a = ask(question)

# PCTF prompt
response_b = ask(
  question,
  system=(
    "You are a senior DevOps engineer mentoring a junior engineer on their first day"
    "Use a single real-world analogy (shipping containers) to ground the explanation"
    "Keep your answer under 80 words."
    "End with one concrete terminal command the junior can try right now"

  )
)

criteria = (
  "Clarity for beginners (0-10)"
  "quality of analogies (0-10)"
  "conciseness - shoter is better (0-10)"
  "actionable next-step the reader can try immediatelly (0-10)"
  "Score each dimension then average for the final score"
)

In [8]:
def evaluate_response(question: str, response: str, criteria: str) -> dict:
    """Have Claude act as an LLM judge and return structured JSON."""
    eval_system = (
        "You are an expert prompt-engineering evaluator.\n"
        "You will receive a QUESTION, a RESPONSE, and CRITERIA.\n\n"
        "Score EACH dimension 1-10, then compute the average as 'score'.\n\n"
        "Return ONLY valid JSON (no markdown fences). Schema:\n"
        '{"score": <float>, "breakdown": {"clarity": <int>, "analogy": <int>, '
        '"conciseness": <int>, "actionable": <int>}, "one_liner": "<= 15 word summary"}'
    )
 
    eval_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"RESPONSE:\n{response}\n\n"
        f"CRITERIA:\n{criteria}"
    )
 
    raw = ask(eval_prompt, system=eval_system, temperature=0.0)
 
    try:
        cleaned = raw.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned.removeprefix("```json").removesuffix("```").strip()
        elif cleaned.startswith("```"):
            cleaned = cleaned.removeprefix("```").removesuffix("```").strip()
 
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"score": 0, "breakdown": {}, "one_liner": "⚠ JSON parse error"}

In [9]:
eval_a = evaluate_response(question, response_a, criteria)
eval_b = evaluate_response(question, response_b, criteria)
 
words_a = len(response_a.split())
words_b = len(response_b.split())
 
winner = "A" if eval_a.get("score", 0) > eval_b.get("score", 0) else "B"
 
md = f"""
# A/B Test Results
 
## Question
**{question}**
 
| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Clarity** | {eval_a.get('breakdown', {}).get('clarity', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('clarity', 'N/A')}/10 |
| **Analogy** | {eval_a.get('breakdown', {}).get('analogy', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('analogy', 'N/A')}/10 |
| **Conciseness** | {eval_a.get('breakdown', {}).get('conciseness', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('conciseness', 'N/A')}/10 |
| **Actionable** | {eval_a.get('breakdown', {}).get('actionable', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('actionable', 'N/A')}/10 |
| **Word Count** | {words_a} | {words_b} |
 
## Prompt A Response
{response_a[:500]}{'...' if len(response_a) > 500 else ''}
 
**Judge summary:** {eval_a.get('one_liner', 'N/A')}
 
---
 
## Prompt B Response
{response_b[:500]}{'...' if len(response_b) > 500 else ''}
 
**Judge summary:** {eval_b.get('one_liner', 'N/A')}
 
---
 
# 🏆 Winner: Prompt {winner}
"""
 
display(Markdown(md))


# A/B Test Results

## Question
**Explain what Docker is to a junior developer**

| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | 7.75/10 | 8.25/10 |
| **Clarity** | 8/10 | 9/10 |
| **Analogy** | 7/10 | 8/10 |
| **Conciseness** | 6/10 | 7/10 |
| **Actionable** | 8/10 | 9/10 |
| **Word Count** | 297 | 62 |

## Prompt A Response
Sure! Docker is a platform that allows developers to automate the deployment, scaling, and management of applications using containers. Here’s a breakdown of what that means:

### 1. **Containers**:
   - **What is a Container?**: Think of a container as a lightweight, portable, and self-sufficient package that includes everything needed to run a piece of software: the code, runtime, libraries, and dependencies. This means you can run your application in the same way on any machine, regardless of...

**Judge summary:** Docker simplifies application deployment using lightweight, portable containers.

---

## Prompt B Response
Think of Docker as a shipping container for your software. Just like shipping containers standardize the way goods are transported, Docker packages your application and its dependencies in a consistent environment. This makes it easy to move and run your application anywhere, without worrying about the underlying system. 

Try running this command to see if Docker is installed:  
```bash
docker --version
```

**Judge summary:** Docker is like a shipping container for your software, ensuring consistency.

---

# 🏆 Winner: Prompt B


---
## 3. Batch Evaluation

Test a prompt across **multiple questions** to measure consistency:

In [10]:
# ── Batch evaluation ──
test_questions = [
    "What is an API?",
    "Explain Git branching.",
    "What is the difference between SQL and NoSQL?",
]

system = "You are a senior developer explaining to a junior. Use an analogy. Max 3 sentences."
criteria = "Accuracy, clarity, appropriate analogy, conciseness"

scores = []
print(f"{'Question':<45} {'Score':>6} {'Words':>6}")
print("─" * 60)

for q in test_questions:
    response = ask(q, system=system)
    evaluation = evaluate_response(q, response, criteria)
    score = evaluation.get('score', 0)
    scores.append(score)
    words = len(response.split())
    print(f"{q[:42]+'...':<45} {score:>5}/10 {words:>5}")

avg = sum(scores) / len(scores) if scores else 0
print(f"\n📊 Average score: {avg:.1f}/10")
print(f"   Score range: {min(scores)}-{max(scores)}/10")
print(f"   Consistency: {'🟢 Good' if max(scores)-min(scores) <= 2 else '🟡 Variable'}")

Question                                       Score  Words
────────────────────────────────────────────────────────────
What is an API?...                             8.25/10    65
Explain Git branching....                      8.25/10    68
What is the difference between SQL and NoS...  8.25/10    80

📊 Average score: 8.2/10
   Score range: 8.25-8.25/10
   Consistency: 🟢 Good


---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **LLM-as-Judge** | Use an LLM to score other LLM outputs |
| **A/B testing** | Compare prompt variants on same questions |
| **Batch evaluation** | Test across multiple inputs for consistency |
| **Scoring rubric** | Define clear criteria (accuracy, clarity, etc.) |
| **JSON output** | Have the judge return structured scores |

---
**Next:** `06_output_parsing.ipynb` — Parse and structure LLM outputs reliably